# Export Projected DEM Matrices And Low-Weight Nulls From Stim Circuits

This notebook exports `.npz` files from representative `.stim` circuits using the same projected Z-basis decoder path as the cluster runners.

The semantics are intentionally **not CSS-data-block semantics**:

- `Hz`: projected DEM check matrix actually used by the decoder
- `Lz`: projected DEM logical-observable matrix actually used by the decoder
- `Hx`: a **heuristic family of low-weight independent null fault combinations** inside `ker([Hz; Lz])`
- `Lx`: unused placeholder with zero rows

Important considerations:

1. These `Hx` rows are **not ordinary code stabilizers on the data block**.
   They live in the **fault-mechanism space** of the projected DEM model.

2. The full nullspace `ker([Hz; Lz])` is typically huge and a raw elimination basis is not meaningful for the half-row experiments.
   A mathematically valid basis can still be very dense.

3. Therefore, this notebook does **not** export a full nullspace basis.
   Instead it searches for **many low-weight nulls**, then greedily keeps an **independent** subset up to a requested cap.

4. The low-weight-null search is heuristic:
   - compute a row-reduced description of `[Hz; Lz]`
   - generate canonical nullspace seed vectors from free columns
   - keep only seeds below a seed-weight cap
   - xor pairs of the lightest seed vectors to discover additional low-weight nulls
   - greedily keep independent candidates whose weight is at most `null_weight_cap`

5. This is a practical compromise.
   It is not guaranteed to find an optimal minimum-weight basis of the nullspace.
   It is intended to produce a more meaningful experiment family than the raw elimination basis.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import stim

from relay_bp.stim.sinter.check_matrices import CheckMatrices
from relay_bp.stim.sinter.runner import _filter_detectors_by_basis

pd.set_option("display.max_colwidth", 200)
np.set_printoptions(edgeitems=20, linewidth=160, suppress=True)

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tests" / "testdata").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}.")


def resolve_unique_stim_path(repo_root: Path, stim_glob: str) -> Path:
    matches = sorted(repo_root.glob(stim_glob))
    if not matches:
        raise FileNotFoundError(f"No files matched {stim_glob!r} under {repo_root}.")
    if len(matches) > 1:
        raise ValueError(f"Expected exactly one match for {stim_glob!r}, got {len(matches)}.")
    return matches[0]


def sparse_to_dense_binary(matrix) -> np.ndarray:
    return np.asarray(matrix.toarray(), dtype=np.uint8) % 2


def bias_array_or_empty(values: np.ndarray | None) -> np.ndarray:
    if values is None:
        return np.zeros((0,), dtype=np.uint8)
    return np.asarray(values, dtype=np.uint8).reshape(-1)


def sparse_rows_to_bitints(matrix) -> list[int]:
    matrix = matrix.tocsr().astype(np.uint8)
    rows = []
    for row_idx in range(matrix.shape[0]):
        start = int(matrix.indptr[row_idx])
        end = int(matrix.indptr[row_idx + 1])
        bits = 0
        for col_idx in matrix.indices[start:end]:
            bits |= 1 << int(col_idx)
        if bits:
            rows.append(bits)
    return rows


def rref_rows_and_pivots(check_matrix, observables_matrix) -> tuple[list[int], list[int], int]:
    rows = sparse_rows_to_bitints(check_matrix) + sparse_rows_to_bitints(observables_matrix)
    n_cols = int(check_matrix.shape[1])
    row_count = len(rows)
    pivot_cols = []
    rank = 0
    for col in range(n_cols):
        mask = 1 << col
        pivot = None
        for row_idx in range(rank, row_count):
            if rows[row_idx] & mask:
                pivot = row_idx
                break
        if pivot is None:
            continue
        if pivot != rank:
            rows[rank], rows[pivot] = rows[pivot], rows[rank]
        pivot_row = rows[rank]
        for row_idx in range(row_count):
            if row_idx != rank and (rows[row_idx] & mask):
                rows[row_idx] ^= pivot_row
        pivot_cols.append(col)
        rank += 1
        if rank == row_count:
            break
    return rows[:rank], pivot_cols, n_cols


def canonical_null_seed_rows(rref_rows: list[int], pivot_cols: list[int], n_cols: int, seed_weight_cap: int) -> list[int]:
    pivot_set = set(pivot_cols)
    free_cols = [col for col in range(n_cols) if col not in pivot_set]
    seeds = []
    for free_col in free_cols:
        vec = 1 << free_col
        mask = 1 << free_col
        for row_bits, pivot_col in zip(rref_rows, pivot_cols):
            if row_bits & mask:
                vec |= 1 << pivot_col
        if vec.bit_count() <= seed_weight_cap:
            seeds.append(vec)
    return sorted(set(seeds), key=lambda vec: (vec.bit_count(), vec))


def select_independent_candidates(candidates: list[int], max_rows: int | None) -> list[int]:
    basis_by_lead = {}
    selected = []
    for vec in sorted(candidates, key=lambda item: (item.bit_count(), item)):
        reduced = vec
        while reduced:
            lead = reduced.bit_length() - 1
            if lead in basis_by_lead:
                reduced ^= basis_by_lead[lead]
            else:
                basis_by_lead[lead] = reduced
                selected.append(vec)
                break
        if max_rows is not None and len(selected) >= max_rows:
            break
    return selected


def bitints_to_dense_rows(bitints: list[int], n_cols: int) -> np.ndarray:
    if not bitints:
        return np.zeros((0, n_cols), dtype=np.uint8)
    packed_width = (n_cols + 7) // 8
    dense = np.zeros((len(bitints), n_cols), dtype=np.uint8)
    for row_idx, value in enumerate(bitints):
        packed = np.frombuffer(value.to_bytes(packed_width, byteorder="little", signed=False), dtype=np.uint8)
        dense[row_idx] = np.unpackbits(packed, bitorder="little")[:n_cols]
    return dense


def find_low_weight_independent_nulls(check_matrix, observables_matrix, *, null_weight_cap: int, seed_weight_cap: int, pairwise_seed_limit: int, max_exported_hx_rows: int | None) -> tuple[np.ndarray, dict[str, int | float]]:
    rref_rows, pivot_cols, n_cols = rref_rows_and_pivots(check_matrix, observables_matrix)
    seed_rows = canonical_null_seed_rows(rref_rows, pivot_cols, n_cols, seed_weight_cap)
    pairwise_seed_rows = seed_rows[: min(pairwise_seed_limit, len(seed_rows))]
    low_weight_candidates = {vec for vec in seed_rows if vec.bit_count() <= null_weight_cap and vec.bit_count() > 0 and (vec.bit_count() % 2 == 0)}
    for left_idx in range(len(pairwise_seed_rows)):
        left = pairwise_seed_rows[left_idx]
        for right_idx in range(left_idx + 1, len(pairwise_seed_rows)):
            candidate = left ^ pairwise_seed_rows[right_idx]
            weight = candidate.bit_count()
            if weight > 0 and weight <= null_weight_cap and (weight % 2 == 0):
                low_weight_candidates.add(candidate)
    selected = select_independent_candidates(list(low_weight_candidates), max_rows=max_exported_hx_rows)
    hx_dense = bitints_to_dense_rows(selected, n_cols)
    exported_weights = hx_dense.sum(axis=1).astype(int) if len(hx_dense) else np.zeros((0,), dtype=int)
    stats = {
        "stacked_rank": int(len(pivot_cols)),
        "stacked_nullity": int(n_cols - len(pivot_cols)),
        "seed_rows_under_seed_cap": int(len(seed_rows)),
        "pairwise_seed_rows_used": int(len(pairwise_seed_rows)),
        "unique_even_candidates_under_null_cap": int(len(low_weight_candidates)),
        "exported_independent_rows": int(hx_dense.shape[0]),
        "exported_min_weight": int(exported_weights.min()) if len(exported_weights) else -1,
        "exported_median_weight": float(np.median(exported_weights)) if len(exported_weights) else -1.0,
        "exported_max_weight": int(exported_weights.max()) if len(exported_weights) else -1,
    }
    return hx_dense, stats


def compute_projected_target(stim_path: Path, *, basis_filter: str, decomposed_hyperedges: bool | None, prune_decided_errors: bool, threshold: float) -> dict[str, object]:
    full_circuit = stim.Circuit.from_file(stim_path)
    filtered_circuit = _filter_detectors_by_basis(full_circuit, basis_filter)
    dem = filtered_circuit.detector_error_model()
    check_matrices = CheckMatrices.from_dem(dem, decomposed_hyperedges=decomposed_hyperedges, prune_decided_errors=prune_decided_errors, threshold=threshold)
    return {
        "stim_path": stim_path,
        "basis_filter": basis_filter,
        "filtered_circuit": filtered_circuit,
        "dem": dem,
        "check_matrices": check_matrices,
        "Hz": sparse_to_dense_binary(check_matrices.check_matrix),
        "Lz": sparse_to_dense_binary(check_matrices.observables_matrix),
    }


def build_payload(*, target_name: str, projected: dict[str, object], hx_dense: np.ndarray, null_stats: dict[str, int | float], null_weight_cap: int, seed_weight_cap: int, pairwise_seed_limit: int, max_exported_hx_rows: int | None) -> dict[str, np.ndarray]:
    check_matrices = projected["check_matrices"]
    dem = projected["dem"]
    metadata = {
        "target_name": target_name,
        "stim_path": str(projected["stim_path"]),
        "basis_filter_applied": str(projected["basis_filter"]),
        "projection_kind": "fault_mechanism_to_Z_sector",
        "Hz_semantics": "projected_check_matrix_used_by_decoder",
        "Lz_semantics": "projected_logical_matrix_used_by_decoder",
        "Hx_semantics": "heuristic_low_weight_independent_nulls_in_kernel_of_[Hz;Lz]",
        "notes": [
            "Hx rows are fault-mechanism null combinations, not data-block stabilizers.",
            "Hx is not a full nullspace basis.",
            "Hx is a heuristic low-weight independent family used for experiments.",
            "The search uses canonical nullspace seeds from a row-reduced representation, plus pairwise xor expansion among the lightest seeds.",
        ],
        "null_search": {
            "null_weight_cap": int(null_weight_cap),
            "seed_weight_cap": int(seed_weight_cap),
            "pairwise_seed_limit": int(pairwise_seed_limit),
            "max_exported_hx_rows": None if max_exported_hx_rows is None else int(max_exported_hx_rows),
            **null_stats,
        },
        "Hz_shape": [int(x) for x in projected["Hz"].shape],
        "Lz_shape": [int(x) for x in projected["Lz"].shape],
        "Hx_shape": [int(x) for x in hx_dense.shape],
        "num_detectors_in_dem": int(dem.num_detectors),
        "num_observables_in_dem": int(dem.num_observables),
        "num_fault_mechanisms_after_pruning": int(projected["Hz"].shape[1]),
    }
    return {
        "Hx": np.asarray(hx_dense, dtype=np.uint8),
        "Hz": np.asarray(projected["Hz"], dtype=np.uint8),
        "Lx": np.zeros((0, projected["Hz"].shape[1]), dtype=np.uint8),
        "Lz": np.asarray(projected["Lz"], dtype=np.uint8),
        "error_priors": np.asarray(check_matrices.error_priors, dtype=np.float64),
        "syndrome_bias": bias_array_or_empty(check_matrices.syndrome_bias),
        "observables_bias": bias_array_or_empty(check_matrices.observables_bias),
        "dem_text": np.asarray(str(dem)),
        "metadata_json": np.asarray(json.dumps(metadata, sort_keys=True)),
    }


def inspect_export(npz_path: Path) -> pd.DataFrame:
    data = np.load(npz_path, allow_pickle=True)
    rows = []
    for key in data.files:
        arr = data[key]
        rows.append({"key": key, "shape": tuple(int(x) for x in arr.shape), "dtype": str(arr.dtype)})
    return pd.DataFrame(rows)

In [ ]:
repo_root = find_repo_root(Path.cwd().resolve())
export_dir = repo_root / "examples" / "notebook_data" / "stim_decoder_npz_exports"
export_dir.mkdir(parents=True, exist_ok=True)

decomposed_hyperedges = None
prune_decided_errors = True
threshold = 0.0
save_files = True

null_weight_cap = 8
seed_weight_cap = 12
pairwise_seed_limit = 512
max_exported_hx_rows = 256

TARGETS = {
    "gross": {
        "stim_glob": "tests/testdata/bicycle_bivariate/circuit=bicycle_bivariate_144_12_12_memory_Z,distance=12,rounds=12,error_rate=0.001,*.stim",
        "basis_filter": "Z",
        "output_name": "gross_from_stim_Z_basis_fault_projection.npz",
    },
    "two_gross": {
        "stim_glob": "tests/testdata/bicycle_bivariate/circuit=bicycle_bivariate_288_12_18_memory_Z,distance=18,rounds=18,error_rate=0.001,*.stim",
        "basis_filter": "Z",
        "output_name": "two_gross_from_stim_Z_basis_fault_projection.npz",
    },
    "surface": {
        "stim_glob": "tests/testdata/surface/circuit=rotated_surface_code_memory_Z,distance=13,rounds=13,error_rate=0.001,*.stim",
        "basis_filter": "Z",
        "output_name": "surface13_from_stim_Z_basis_fault_projection.npz",
    },
}

display(pd.DataFrame([
    {
        "target_name": name,
        "stim_glob": target["stim_glob"],
        "basis_filter": target["basis_filter"],
        "output_name": target["output_name"],
        "null_weight_cap": null_weight_cap,
        "seed_weight_cap": seed_weight_cap,
        "pairwise_seed_limit": pairwise_seed_limit,
        "max_exported_hx_rows": max_exported_hx_rows,
    }
    for name, target in TARGETS.items()
]))

## Gross: Resolve Circuit

This is a separate step on purpose. We want the notebook to make it explicit which exact circuit file is being used before any matrix extraction happens.

In [ ]:
gross_target = TARGETS["gross"]
gross_stim_path = resolve_unique_stim_path(repo_root, gross_target["stim_glob"])
display(pd.DataFrame([{
    "target_name": "gross",
    "stim_path": str(gross_stim_path),
    "basis_filter": gross_target["basis_filter"],
    "output_path": str(export_dir / gross_target["output_name"]),
}]))

## Gross: Build Projected `Hz` And `Lz`

This reproduces the same projected decoder object used by the Z-basis runner: load Stim circuit -> apply basis filter -> build DEM -> extract decoder matrices.

In [ ]:
gross_projected = compute_projected_target(
    gross_stim_path,
    basis_filter=gross_target["basis_filter"],
    decomposed_hyperedges=decomposed_hyperedges,
    prune_decided_errors=prune_decided_errors,
    threshold=threshold,
)
display(pd.DataFrame([{
    "target_name": "gross",
    "Hz_shape": gross_projected["Hz"].shape,
    "Lz_shape": gross_projected["Lz"].shape,
    "n_error_priors": int(gross_projected["check_matrices"].error_priors.shape[0]),
    "has_syndrome_bias": bool(gross_projected["check_matrices"].syndrome_bias is not None),
    "has_observables_bias": bool(gross_projected["check_matrices"].observables_bias is not None),
}]))

## Gross: Search Low-Weight Independent Nulls For `Hx`

This is the key heuristic step. We are **not** exporting a full nullspace basis. We are exporting a capped family of independent low-weight nulls that is more suitable for the later half-support experiments.

In [ ]:
gross_hx, gross_null_stats = find_low_weight_independent_nulls(
    gross_projected["check_matrices"].check_matrix,
    gross_projected["check_matrices"].observables_matrix,
    null_weight_cap=null_weight_cap,
    seed_weight_cap=seed_weight_cap,
    pairwise_seed_limit=pairwise_seed_limit,
    max_exported_hx_rows=max_exported_hx_rows,
)
display(pd.DataFrame([{"target_name": "gross", "Hx_shape": gross_hx.shape, **gross_null_stats}]))

## Gross: Write Export

The saved `Hx` now means ?heuristic low-weight independent null family in the projected fault model?. The metadata records the search parameters and warns about the non-CSS semantics.

In [ ]:
gross_output_path = export_dir / gross_target["output_name"]
gross_payload = build_payload(
    target_name="gross",
    projected=gross_projected,
    hx_dense=gross_hx,
    null_stats=gross_null_stats,
    null_weight_cap=null_weight_cap,
    seed_weight_cap=seed_weight_cap,
    pairwise_seed_limit=pairwise_seed_limit,
    max_exported_hx_rows=max_exported_hx_rows,
)
if save_files:
    np.savez_compressed(gross_output_path, **gross_payload)
display(inspect_export(gross_output_path))

## Two Gross: Resolve Circuit

In [ ]:
two_gross_target = TARGETS["two_gross"]
two_gross_stim_path = resolve_unique_stim_path(repo_root, two_gross_target["stim_glob"])
display(pd.DataFrame([{
    "target_name": "two_gross",
    "stim_path": str(two_gross_stim_path),
    "basis_filter": two_gross_target["basis_filter"],
    "output_path": str(export_dir / two_gross_target["output_name"]),
}]))

## Two Gross: Build Projected `Hz` And `Lz`

In [ ]:
two_gross_projected = compute_projected_target(
    two_gross_stim_path,
    basis_filter=two_gross_target["basis_filter"],
    decomposed_hyperedges=decomposed_hyperedges,
    prune_decided_errors=prune_decided_errors,
    threshold=threshold,
)
display(pd.DataFrame([{
    "target_name": "two_gross",
    "Hz_shape": two_gross_projected["Hz"].shape,
    "Lz_shape": two_gross_projected["Lz"].shape,
    "n_error_priors": int(two_gross_projected["check_matrices"].error_priors.shape[0]),
    "has_syndrome_bias": bool(two_gross_projected["check_matrices"].syndrome_bias is not None),
    "has_observables_bias": bool(two_gross_projected["check_matrices"].observables_bias is not None),
}]))

## Two Gross: Search Low-Weight Independent Nulls For `Hx`

In [ ]:
two_gross_hx, two_gross_null_stats = find_low_weight_independent_nulls(
    two_gross_projected["check_matrices"].check_matrix,
    two_gross_projected["check_matrices"].observables_matrix,
    null_weight_cap=null_weight_cap,
    seed_weight_cap=seed_weight_cap,
    pairwise_seed_limit=pairwise_seed_limit,
    max_exported_hx_rows=max_exported_hx_rows,
)
display(pd.DataFrame([{"target_name": "two_gross", "Hx_shape": two_gross_hx.shape, **two_gross_null_stats}]))

## Two Gross: Write Export

In [ ]:
two_gross_output_path = export_dir / two_gross_target["output_name"]
two_gross_payload = build_payload(
    target_name="two_gross",
    projected=two_gross_projected,
    hx_dense=two_gross_hx,
    null_stats=two_gross_null_stats,
    null_weight_cap=null_weight_cap,
    seed_weight_cap=seed_weight_cap,
    pairwise_seed_limit=pairwise_seed_limit,
    max_exported_hx_rows=max_exported_hx_rows,
)
if save_files:
    np.savez_compressed(two_gross_output_path, **two_gross_payload)
display(inspect_export(two_gross_output_path))

## Surface: Resolve Circuit

In [ ]:
surface_target = TARGETS["surface"]
surface_stim_path = resolve_unique_stim_path(repo_root, surface_target["stim_glob"])
display(pd.DataFrame([{
    "target_name": "surface",
    "stim_path": str(surface_stim_path),
    "basis_filter": surface_target["basis_filter"],
    "output_path": str(export_dir / surface_target["output_name"]),
}]))

## Surface: Build Projected `Hz` And `Lz`

In [ ]:
surface_projected = compute_projected_target(
    surface_stim_path,
    basis_filter=surface_target["basis_filter"],
    decomposed_hyperedges=decomposed_hyperedges,
    prune_decided_errors=prune_decided_errors,
    threshold=threshold,
)
display(pd.DataFrame([{
    "target_name": "surface",
    "Hz_shape": surface_projected["Hz"].shape,
    "Lz_shape": surface_projected["Lz"].shape,
    "n_error_priors": int(surface_projected["check_matrices"].error_priors.shape[0]),
    "has_syndrome_bias": bool(surface_projected["check_matrices"].syndrome_bias is not None),
    "has_observables_bias": bool(surface_projected["check_matrices"].observables_bias is not None),
}]))

## Surface: Search Low-Weight Independent Nulls For `Hx`

In [ ]:
surface_hx, surface_null_stats = find_low_weight_independent_nulls(
    surface_projected["check_matrices"].check_matrix,
    surface_projected["check_matrices"].observables_matrix,
    null_weight_cap=null_weight_cap,
    seed_weight_cap=seed_weight_cap,
    pairwise_seed_limit=pairwise_seed_limit,
    max_exported_hx_rows=max_exported_hx_rows,
)
display(pd.DataFrame([{"target_name": "surface", "Hx_shape": surface_hx.shape, **surface_null_stats}]))

## Surface: Write Export

In [ ]:
surface_output_path = export_dir / surface_target["output_name"]
surface_payload = build_payload(
    target_name="surface",
    projected=surface_projected,
    hx_dense=surface_hx,
    null_stats=surface_null_stats,
    null_weight_cap=null_weight_cap,
    seed_weight_cap=seed_weight_cap,
    pairwise_seed_limit=pairwise_seed_limit,
    max_exported_hx_rows=max_exported_hx_rows,
)
if save_files:
    np.savez_compressed(surface_output_path, **surface_payload)
display(inspect_export(surface_output_path))